In [2]:
import os
import time
import hashlib
import json
import requests
from datetime import datetime, timezone
from confluent_kafka import Producer
from pathlib import Path
import threading
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

In [3]:
'''
    This notebook contains code to produce messages to Kafka topics from multiple CSV files.
    The ConfluentMultiFileBroadcaster class reads CSV files from a specified directory,
    processes each row, and sends the data as messages to a Kafka topic using the Confluent Kafka Producer.

    The main intention of the project was to have real time news streaming from various news sources using newsapi.org.
    However, due to limitations with the free tier of the newsapi.org, we simulated the streaming using pre-downloaded CSV files.
    The code for reading from newsapi.org is included but commented out.

    Key Features:
    - Reads multiple CSV files from a given directory.
    - Sends each row as a message to specified Kafka topics.
    - Supports configurable chunk sizes and multi-threaded message production.
    - Provides progress indication
'''

'\n    This notebook contains code to produce messages to Kafka topics from multiple CSV files.\n    The ConfluentMultiFileBroadcaster class reads CSV files from a specified directory,\n    processes each row, and sends the data as messages to a Kafka topic using the Confluent Kafka Producer.\n\n    The main intention of the project was to have real time news streaming from various news sources using newsapi.org.\n    However, due to limitations with the free tier of the newsapi.org, we simulated the streaming using pre-downloaded CSV files.\n    The code for reading from newsapi.org is included but commented out.\n\n    Key Features:\n    - Reads multiple CSV files from a given directory.\n    - Sends each row as a message to specified Kafka topics.\n    - Supports configurable chunk sizes and multi-threaded message production.\n    - Provides progress indication\n'

In [9]:
#We define the data directory containing the CSV files and randomly selected 25 stocks for streaming
DATA_DIR = Path("C:/Users/rahul/OneDrive/7_Learning/IISC/Courses/4.2_Data_Engineering_at_Scale/Course Material/Codes/Project/Streaming/data/engineered_data_enhanced")

selected_25 = ['amgn.csv','TSM.csv','cmg.csv','orcl.csv','GE.csv','cmcsa.csv','pypl.csv','ebay.csv','biib.csv',
        'qcom.csv','AMD.csv','COST.csv','crm.csv','BABA.csv','cop.csv','CVX.csv','uso.csv','nke.csv','WFC.csv',
        'mrk.csv','aal.csv','gsk.csv','QQQ.csv','pep.csv','ABBV.csv']
selected = [x.upper() for x in selected_25]

selected_stock_paths = [DATA_DIR / s for s in selected]
selected_topic_names = ['NASDAQ_' + s.replace('.CSV', '') for s in selected]
selected_stock_names = [s.replace('.CSV', '') for s in selected]

In [6]:
#Code to read from the pre-downloaded CSV files and produce messages to Kafka topics
class ConfluentMultiFileBroadcaster:
    def __init__(self,
                 file_paths,
                 topics,
                 bootstrap_servers="localhost:9092",
                 chunk_size=1000,
                 worker_threads=4,
                 key_field=None,
                 producer_conf_extra=None,
                 status_print_every_n_chunks=5):
        self.file_paths = [Path(p) for p in file_paths]
        self.topics = topics
        self.chunk_size = int(chunk_size)
        self.key_field = key_field
        self._running = False
        self._thread = None
        self._executor = ThreadPoolExecutor(max_workers=worker_threads)
        self._futures = []
        self.status_print_every_n_chunks = int(status_print_every_n_chunks)

        conf = {
            "bootstrap.servers": bootstrap_servers,
            "acks": "all",
            "enable.idempotence": True,
            "retries": 10,
            "linger.ms": 50,
            "compression.type": "lz4",
            "queue.buffering.max.messages": 100000,
            "batch.num.messages": 10000,
        }
        if producer_conf_extra:
            conf.update(producer_conf_extra)
        self._producer = Producer(conf)

    def _delivery(self, err, msg):
        if err:
            print(f"[delivery error] {err}")

    def _send_batch(self, records):
        for rec in records:
            key = None
            if self.key_field and self.key_field in rec:
                key = str(rec[self.key_field]).encode("utf-8")
            val = json.dumps(rec).encode("utf-8")
            for t in self.topics:
                try:
                    self._producer.produce(topic=t, value=val, key=key, callback=self._delivery)
                except Exception as e:
                    # queue full or other transient issue
                    print("Produce error:", e)
            # let librdkafka process callbacks
            self._producer.poll(0)
        # final quick poll
        self._producer.poll(0)

    def start(self, show_progress=True):
        if self._running:
            print("Already running")
            return
        self._running = True
        self._thread = threading.Thread(target=self._run_loop, args=(show_progress,), daemon=True)
        self._thread.start()
        print("Producer started in background.")

    def stop(self, flush=True, timeout=30):
        if not self._running:
            print("Not running")
            return
        print("Stopping producer…")
        self._running = False
        if self._thread:
            self._thread.join(timeout=10)
        for fut in self._futures:
            try:
                fut.result(timeout=5)
            except:
                pass
        if flush:
            print("Flushing producer…")
            self._producer.flush(timeout)
        print("Producer stopped.")

    def _run_loop(self, show_progress):
        try:
            for fp in self.file_paths:
                if not self._running:
                    break

                # try to get an exact row count; if unavailable, set to None
                try:
                    total_rows = sum(1 for _ in open(fp)) - 1
                    if total_rows < 0:
                        total_rows = None
                except Exception:
                    total_rows = None

                # If we have a reliable total, use tqdm if available. Otherwise fallback to chunk counters.
                use_tqdm = False
                pbar = None
                if show_progress and total_rows is not None:
                    try:
                        from tqdm import tqdm
                        pbar = tqdm(total=total_rows, desc=f"Producing {fp.name}")
                        use_tqdm = True
                    except Exception:
                        pbar = None
                        use_tqdm = False

                # fallback: print basic status every few chunks
                chunk_counter = 0
                rows_sent_for_file = 0
                start_time = time.time()

                for chunk in pd.read_csv(fp, chunksize=self.chunk_size):
                    if not self._running:
                        break
                    records = chunk.to_dict(orient="records")
                    fut = self._executor.submit(self._send_batch, records)
                    self._futures.append(fut)
                    # prune done futures
                    self._futures = [f for f in self._futures if not f.done()]

                    # update counters and UI
                    chunk_counter += 1
                    rows_sent_for_file += len(records)
                    if use_tqdm and pbar:
                        try:
                            pbar.update(len(records))
                        except Exception:
                            # if tqdm fails for some reason, disable it for rest of this file
                            try:
                                pbar.close()
                            except Exception:
                                pass
                            pbar = None
                            use_tqdm = False
                    else:
                        # print a lightweight status every N chunks
                        if (chunk_counter % self.status_print_every_n_chunks) == 0:
                            elapsed = time.time() - start_time
                            rows_per_sec = rows_sent_for_file / elapsed if elapsed > 0 else 0
                            print(f"[{fp.name}] chunks={chunk_counter} rows={rows_sent_for_file} avg_rows/s={rows_per_sec:.1f}")

                # file done
                if pbar:
                    try:
                        pbar.close()
                    except Exception:
                        pass
                else:
                    # final summary for file if we didn't use tqdm
                    elapsed = time.time() - start_time
                    rows_per_sec = rows_sent_for_file / elapsed if elapsed > 0 else 0
                    print(f"[{fp.name}] completed. total_rows_sent={rows_sent_for_file} time_s={elapsed:.1f} avg_rows/s={rows_per_sec:.1f}")

                # small pause between files (optional, helps IO)
                time.sleep(0.1)

        except Exception as e:
            print("Error in producer loop:", e)
        finally:
            try:
                self._producer.poll(0)
            except Exception:
                pass


In [ ]:
#Using the ConfluentMultiFileBroadcaster to produce messages from selected CSV files to Kafka topics
TOPICS = selected_topic_names
BOOT = "localhost:9092"

b = ConfluentMultiFileBroadcaster(
    file_paths=selected_stock_paths,
    topics=TOPICS,
    bootstrap_servers=BOOT,
    chunk_size=1000,
    worker_threads=4,
    key_field="symbol"
)

b.start(show_progress=True)

In [ ]:
NEWSAPI_KEY = 'YOUR_NEWSAPI_KEY_HERE'  # Replace with your NewsAPI key

In [ ]:
#Code for reading from newsapi.org and producing messages to Kafka topics
#This code is commented out due to limitations with the free tier of newsapi.org and volume of requests

import time
import hashlib
import json
import os
import requests
from confluent_kafka import Producer

NEWS_API_KEY = os.environ.get("NEWS_API_KEY")  # set in notebook or OS
KAFKA_BOOTSTRAP = os.environ.get("KAFKA_BOOTSTRAP", "localhost:9092")
TOPIC_PREFIX = os.environ.get("TOPIC_PREFIX", "NASDAQ_")  # new: prefix for per-stock topics
POLL_INTERVAL = int(os.environ.get("POLL_INTERVAL", 10))

STOCK_QUERIES = {
    "AMGN": "Amgen OR AMGN OR \"Amgen Inc\"",
    "TSM": "TSMC OR TSM OR \"Taiwan Semiconductor\"",
    "CMG": "Chipotle OR CMG OR \"Chipotle Mexican Grill\"",
    "ORCL": "Oracle OR ORCL OR \"Oracle Corporation\"",
    "GE": "General Electric OR GE OR \"GE Aerospace\"",
    "CMCSA": "Comcast OR CMCSA OR \"Comcast Corporation\"",
    "PYPL": "PayPal OR PYPL OR \"PayPal Holdings\"",
    "EBAY": "eBay OR EBAY OR \"eBay Inc\"",
    "BIIB": "Biogen OR BIIB OR \"Biogen Inc\"",
    "QCOM": "Qualcomm OR QCOM OR \"Qualcomm Incorporated\"",
    "AMD": "AMD OR \"Advanced Micro Devices\" OR AMD",
    "COST": "Costco OR COST OR \"Costco Wholesale\"",
    "CRM": "Salesforce OR CRM OR \"Salesforce Inc\"",
    "BABA": "Alibaba OR BABA OR \"Alibaba Group\"",
    "COP": "ConocoPhillips OR COP OR \"Conoco Phillips\"",
    "CVX": "Chevron OR CVX OR \"Chevron Corporation\"",
    "USO": "US Oil Fund OR USO OR \"United States Oil Fund\"",
    "NKE": "Nike OR NKE OR \"Nike Inc\"",
    "WFC": "Wells Fargo OR WFC OR \"Wells Fargo & Company\"",
    "MRK": "Merck OR MRK OR \"Merck & Co\"",
    "AAL": "American Airlines OR AAL OR \"American Airlines Group\"",
    "GSK": "GSK OR GlaxoSmithKline OR \"GSK plc\"",
    "QQQ": "Invesco QQQ OR QQQ OR \"NASDAQ 100 ETF\"",
    "PEP": "Pepsi OR PEP OR \"PepsiCo Inc\"",
    "ABBV": "AbbVie OR ABBV OR \"AbbVie Inc\"",
}

DEDUP_DIR = "./dedupe_ids"
os.makedirs(DEDUP_DIR, exist_ok=True)

def make_article_id(article):
    key = (article.get("url") or "") + (article.get("title") or "") + (article.get("publishedAt") or "")
    return hashlib.sha1(key.encode()).hexdigest()

def load_last_seen(symbol):
    f = os.path.join(DEDUP_DIR, f"{symbol}.txt")
    if not os.path.exists(f):
        return set()
    return set(open(f).read().splitlines())

def save_last_seen(symbol, ids):
    f = os.path.join(DEDUP_DIR, f"{symbol}.txt")
    with open(f, "w") as fp:
        for i in ids:
            fp.write(i + "\n")

def fetch_news_sync(query):
    url = "https://newsapi.org/v2/everything"
    params = {
        "q": query,
        "pageSize": 20,
        "sortBy": "publishedAt",
        "language": "en",
        "apiKey": NEWS_API_KEY,
    }
    r = requests.get(url, params=params, timeout=15)
    r.raise_for_status()
    return r.json()

def delivery_report(err, msg):
    if err is not None:
        print("Delivery failed:", err)
    else:
        print(f"Produced to {msg.topic()} [{msg.partition()}] @ {msg.offset()}")

conf = {
    "bootstrap.servers": KAFKA_BOOTSTRAP,
    "linger.ms": 5,
    "compression.type": "snappy",
}
producer = Producer(conf)

try:
    counter = 0
    while counter < 3:
        print("Starting cycle", counter + 1)
        counter += 1
        for symbol, query in STOCK_QUERIES.items():
            last_seen = load_last_seen(symbol)
            new_seen = set()
            try:
                data = fetch_news_sync(query)
                print(f"[{symbol}] fetched {len(data.get('articles', []))} articles")
            except Exception as e:
                print(f"[{symbol}] fetch error:", e)
                continue

            # topic for this stock
            topic_name = f"{TOPIC_PREFIX}{symbol}"

            for a in data.get("articles", []) or []:
                article_id = make_article_id(a)
                if article_id in last_seen:
                    continue
                message = {
                    "article_id": article_id,
                    "symbol": symbol,
                    "title": a.get("title"),
                    "description": a.get("description"),
                    "content": a.get("content"),
                    "url": a.get("url"),
                    "published_at": a.get("publishedAt"),
                    "source": a.get("source", {}).get("name", "newsapi"),
                }
                try:
                    producer.produce(
                        topic_name,
                        key=symbol.encode(),
                        value=json.dumps(message).encode(),
                        callback=delivery_report,
                    )
                    print(f"[{symbol}] produced article_id={article_id} -> topic={topic_name}")
                except BufferError:
                    # buffer full: poll to flush delivery callbacks and retry once
                    producer.poll(0.1)
                    producer.produce(
                        topic_name,
                        key=symbol.encode(),
                        value=json.dumps(message).encode(),
                        callback=delivery_report,
                    )
                new_seen.add(article_id)

            # flush per-symbol to ensure delivery before saving dedupe state
            producer.flush()
            save_last_seen(symbol, load_last_seen(symbol).union(new_seen))

        print("Cycle complete — sleeping", POLL_INTERVAL, "seconds")
        time.sleep(POLL_INTERVAL)
except KeyboardInterrupt:
    print("Stopping producer")
finally:
    producer.flush()
